# Chapter 15 — Instrumental Variables & 2SLS
## University of Neuchâtel — Python for Econometrics

Topics:
1. Endogeneity and why OLS is biased
2. IV estimator (simple case)
3. 2SLS: Stage 1 and Stage 2 manually
4. 2SLS with linearmodels
5. Instrument validity: relevance and exogeneity
6. Wu-Hausman endogeneity test

**Format:** One cell per step. Manual calculation first, then statsmodels/linearmodels, then `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import wooldridge as woo
import os
try:
    from linearmodels.iv import IV2SLS
    HAS_LM = True
except ImportError:
    HAS_LM = False
    print('linearmodels not installed')

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Endogeneity Problem

**OLS assumption:** $Cov(x, u) = 0$ (no correlation between regressor and error).

**Endogeneity:** $Cov(x, u) \neq 0$ → OLS is **biased and inconsistent**.

**Common causes:**
- Omitted variable bias (e.g. ability omitted from wage equation)
- Simultaneity (supply and demand)
- Measurement error in regressors

**Example:** Returns to education

$$\ln(wage_i) = \beta_0 + \beta_1 educ_i + \beta_2 ability_i + u_i$$

We don't observe $ability_i$, so it's in the error. Since $Cov(educ, ability) > 0$,
OLS overstates $\beta_1$ (upward ability bias).

## 2 — Load Data: `card`

In [ ]:
df = woo.dataWoo('card')
print('Shape:', df.shape)
print(df[['lwage','educ','nearc4','exper','black','smsa']].head(10))

In [ ]:
print('Descriptive statistics:')
print(df[['lwage','educ','nearc4','exper','black']].describe().round(3))
print('\nnearc4 (near 4-yr college):', df['nearc4'].value_counts().to_dict())

## 3 — Naive OLS (ignoring endogeneity)

**Model:** $\ln(wage) = \beta_0 + \beta_1 educ + \beta_2 exper + \beta_3 black + \beta_4 smsa + u$

This likely **overstates** $\beta_1$ because $educ$ and ability (in $u$) are positively correlated.

### Step 3.1 — Manual OLS

In [ ]:
df = df.dropna(subset=['lwage','educ','nearc4','exper','black','smsa'])
n = len(df)
X_ols = np.column_stack([
    np.ones(n),
    df['educ'].values,
    df['exper'].values,
    df['black'].values,
    df['smsa'].values
])
y_arr = df['lwage'].values
beta_ols = np.linalg.solve(X_ols.T @ X_ols, X_ols.T @ y_arr)
labels_ols = ['const','educ','exper','black','smsa']
print('OLS coefficients (manual):')
for lbl, b in zip(labels_ols, beta_ols):
    print(f'  {lbl:8s}: {b:.4f}')

### Step 3.2 — Statsmodels OLS

In [ ]:
res_ols = smf.ols('lwage ~ educ + exper + black + smsa', data=df).fit()
print(res_ols.summary())

### Step 3.3 — Compare OLS manual vs statsmodels

In [ ]:
sm_ols = res_ols.params
print('OLS comparison:')
for lbl, b_man in zip(labels_ols, beta_ols):
    match = np.isclose(b_man, sm_ols[lbl])
    print(f'  {lbl:8s}: manual={b_man:.4f}, sm={sm_ols[lbl]:.4f}, match={match}')

## 4 — Instrumental Variable (IV) Estimator

**Instrument:** $nearc4$ (grew up near a 4-year college)

**Two requirements for a valid instrument:**
1. **Relevance:** $Cov(nearc4, educ) \neq 0$ — college proximity affects education ✓
2. **Exogeneity:** $Cov(nearc4, u) = 0$ — college proximity doesn't affect wages directly ✓ (assumption)

**Simple IV formula (one instrument, one endogenous variable):**

$$\hat{\beta}_{IV} = \frac{Cov(z, y)}{Cov(z, x)} = \frac{\sum(z_i - \bar{z})(y_i - \bar{y})}{\sum(z_i - \bar{z})(x_i - \bar{x})}$$

### Step 4.1 — Check instrument relevance

In [ ]:
corr_z_x = np.corrcoef(df['nearc4'], df['educ'])[0,1]
print(f'Correlation(nearc4, educ): {corr_z_x:.4f}')
res_first = smf.ols('educ ~ nearc4 + exper + black + smsa', data=df).fit()
print(f'First-stage F-stat on nearc4: {res_first.fvalue:.2f}')
print(f'First-stage t-stat on nearc4: {res_first.tvalues["nearc4"]:.4f}')
print(f'(Rule of thumb: F >= 10 for strong instrument; F={res_first.f_test("nearc4=0").fvalue[0][0]:.2f})')

### Step 4.2 — Simple IV estimator manually (bivariate case)

In [ ]:
z = df['nearc4'].values
x = df['educ'].values
y = df['lwage'].values
z_dev = z - z.mean()
x_dev = x - x.mean()
y_dev = y - y.mean()
# Simple IV: cov(z,y)/cov(z,x)
b1_iv_simple = np.sum(z_dev * y_dev) / np.sum(z_dev * x_dev)
b0_iv_simple = y.mean() - b1_iv_simple * x.mean()
print(f'Simple IV estimator (bivariate): b1 = {b1_iv_simple:.4f}')
print(f'OLS estimator (bivariate):       b1 = {np.sum(x_dev*y_dev)/np.sum(x_dev**2):.4f}')
print('IV < OLS suggests upward ability bias in OLS')

## 5 — Two-Stage Least Squares (2SLS)

**2SLS with multiple controls** (correct approach):

**Stage 1:** Regress endogenous variable on instrument + all exogenous variables:

$$educ_i = \pi_0 + \pi_1 nearc4_i + \pi_2 exper_i + \pi_3 black_i + \pi_4 smsa_i + v_i$$

**Stage 2:** Replace $educ$ with $\hat{educ}$ and run OLS:

$$\ln(wage_i) = \beta_0 + \beta_1 \widehat{educ}_i + \beta_2 exper_i + \beta_3 black_i + \beta_4 smsa_i + u_i$$

### Step 5.1 — Stage 1: Regress educ on nearc4 + controls

In [ ]:
res_s1 = smf.ols('educ ~ nearc4 + exper + black + smsa', data=df).fit()
educ_hat = res_s1.fittedvalues
print('Stage 1 results:')
print(res_s1.summary())
print(f'F-stat on instrument: {res_s1.f_test("nearc4=0").fvalue[0][0]:.2f}')

In [ ]:
df['educ_hat'] = educ_hat
print('educ vs educ_hat (first 8 rows):')
print(df[['educ','educ_hat']].head(8).round(4))

### Step 5.2 — Stage 2: OLS of lwage ~ educ_hat + controls

In [ ]:
X_s2 = np.column_stack([
    np.ones(n),
    educ_hat.values,
    df['exper'].values,
    df['black'].values,
    df['smsa'].values
])
beta_2sls_manual = np.linalg.solve(X_s2.T @ X_s2, X_s2.T @ y_arr)
labels_2sls = ['const','educ','exper','black','smsa']
print('2SLS Stage 2 (manual):')
for lbl, b in zip(labels_2sls, beta_2sls_manual):
    print(f'  {lbl:8s}: {b:.4f}')

In [ ]:
res_s2_naive = smf.ols('lwage ~ educ_hat + exper + black + smsa', data=df).fit()
print('Stage 2 OLS (manual educ_hat):')
print('NOTE: SEs from this are WRONG — must use IV2SLS for correct SEs')
print(res_s2_naive.summary())

### Step 5.3 — 2SLS with linearmodels IV2SLS

In [ ]:
if HAS_LM:
    dep = df['lwage']
    exog = sm.add_constant(df[['exper','black','smsa']])
    endog = df[['educ']]
    instr = df[['nearc4']]
    res_iv = IV2SLS(dep, exog, endog, instr).fit(cov_type='unadjusted')
    print(res_iv.summary)
else:
    print('linearmodels not available; using manual stage 2 above')

### Step 5.4 — Compare 2SLS manual vs linearmodels

In [ ]:
if HAS_LM:
    print('2SLS coefficient comparison:')
    lm_params = res_iv.params
    for b_man, lbl in zip(beta_2sls_manual, labels_2sls):
        lm_key = 'const' if lbl=='const' else lbl
        if lm_key in lm_params:
            b_lm = lm_params[lm_key]
            print(f'  {lbl:8s}: manual={b_man:.4f}, IV2SLS={b_lm:.4f}, match={np.isclose(b_man,b_lm)}')
else:
    print('OLS vs 2SLS educ coefficient:')
    print(f'  OLS:  {beta_ols[1]:.4f}')
    print(f'  2SLS: {beta_2sls_manual[1]:.4f}')

### Step 5.5 — OLS vs 2SLS comparison

In [ ]:
print('Return to education: OLS vs 2SLS')
print(f'  OLS  (biased): {res_ols.params["educ"]:.4f}')
print(f'  2SLS (manual): {beta_2sls_manual[1]:.4f}')
diff = beta_2sls_manual[1] - res_ols.params['educ']
print(f'  Difference (2SLS-OLS): {diff:.4f}')
if diff < 0:
    print('  2SLS < OLS: consistent with upward ability bias in OLS')
else:
    print('  2SLS > OLS: instrument may capture different population effect (LATE)')

## 6 — Wu-Hausman Test for Endogeneity

**H0:** `educ` is exogenous (OLS is consistent)
**H1:** `educ` is endogenous (2SLS needed)

**Manual procedure:**
1. Get first-stage residuals $\hat{v}$
2. Regress $lwage$ on $educ$, controls, AND $\hat{v}$
3. $t$-test on $\hat{v}$: significant → endogeneity confirmed

In [ ]:
df['v_hat'] = res_s1.resid
res_hausman = smf.ols('lwage ~ educ + exper + black + smsa + v_hat', data=df).fit()
t_vhat = res_hausman.tvalues['v_hat']
p_vhat = res_hausman.pvalues['v_hat']
print('Wu-Hausman test (control function approach):')
print(f'  t-stat on v_hat: {t_vhat:.4f}')
print(f'  p-value:         {p_vhat:.4f}')
if p_vhat < 0.05:
    print('  Reject H0: educ is endogenous → use 2SLS')
else:
    print('  Fail to reject H0: OLS may be consistent')

## 7 — Summary Table

In [ ]:
sumtab = pd.DataFrame({
    'Estimator': ['OLS','2SLS (manual)'],
    'educ_coeff': [res_ols.params['educ'], beta_2sls_manual[1]],
    'educ_SE':    [res_ols.bse['educ'], np.nan]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch15_ols_vs_iv.csv'), index=False)
print('Saved table_ch15_ols_vs_iv.csv')